# 08b. RQ3 age strand -- forward forecasting (Year 9-16)


## 0. Setup

In [36]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 2026
ALR_EPSILON = 1e-6

FORECAST_START_YEAR = 8   # last observed year, forecasting begins at 9
FORECAST_HORIZON = 8      # Year 9 .. Year 16
YEAR_REF = 8               # keep the same time_trend scale used in training

AGE_DATA_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Shuhan Zhao\q3")
OUTPUT_DIR = Path(r"C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs")
FORECAST_DIR = OUTPUT_DIR / 'forecast_year9_16'
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

best_params_df = pd.read_csv(OUTPUT_DIR / 'best_hyperparameters.csv')
summary = pd.read_csv(OUTPUT_DIR / 'summary_full.csv')

with open(OUTPUT_DIR / 'fitted_models.pkl', 'rb') as f:
    fitted_models = pickle.load(f)

print('tasks available in summary:', summary['task'].unique().tolist())
print('fitted models available:', list(fitted_models.keys()))

tasks available in summary: ['age_activity_level', 'age_days10p60gr', 'age_months12', 'age_overall_level', 'disability_activity_level', 'disability_days10p60gr', 'disability_months12', 'disability_overall_level']
fitted models available: ['age_overall__Ridge Regression', 'age_overall__Random Forest', 'age_overall__Gradient Boosting', 'dis_overall__Ridge Regression', 'dis_overall__Random Forest', 'dis_overall__Gradient Boosting', 'dis_level__Ridge Regression', 'dis_level__Random Forest', 'dis_level__Gradient Boosting', 'months12__Ridge Regression', 'months12__Random Forest', 'months12__Gradient Boosting', 'days__Ridge Regression', 'days__Random Forest', 'days__Gradient Boosting', 'age_months12__Ridge Regression', 'age_months12__Random Forest', 'age_months12__Gradient Boosting', 'age_days__Ridge Regression', 'age_days__Random Forest', 'age_days__Gradient Boosting', 'age_level__Ridge Regression', 'age_level__Random Forest', 'age_level__Gradient Boosting']


## 1. Shared forecasting functions


In [37]:
def alr_to_shares(values):
    values = np.clip(np.asarray(values, dtype=float), -30, 30)
    exponent = np.exp(values)
    denominator = 1 + exponent.sum(axis=1, keepdims=True)
    return np.column_stack([exponent / denominator, 1 / denominator])


def shares_to_alr(shares):
    clipped = np.clip(np.asarray(shares, dtype=float), ALR_EPSILON, 1)
    clipped = clipped / clipped.sum(axis=1, keepdims=True)
    return np.log(clipped[:, :2] / clipped[:, [2]])


def select_best_model(summary_df, task_name, metric_col):
    subset = summary_df[
        (summary_df['task'] == task_name) &
        (summary_df['split'] == 'validation')
    ]
    best_row = subset.loc[subset[metric_col].idxmin()]
    return best_row['model']

In [38]:
def add_lag_features(frame, panel_keys, value_cols, lags=(1,), rolling_window=None, covid_years=(5, 6)):
    prepared = frame.sort_values(panel_keys + ['year']).reset_index(drop=True)
    grouped = prepared.groupby(panel_keys, sort=False)
    for column in value_cols:
        for lag in lags:
            prepared[f'{column}_lag{lag}'] = grouped[column].shift(lag)
        if rolling_window:
            prepared[f'{column}_roll{rolling_window}'] = grouped[column].transform(
                lambda s: s.shift(1).rolling(rolling_window, min_periods=1).mean()
            )
    prepared['time_trend'] = prepared['year'] / prepared['year'].max()
    prepared['is_covid_year'] = prepared['year'].isin(covid_years).astype(int)
    return prepared


def build_model(model_name, parameters, numeric_features, categorical_features, multi_output):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if model_name == 'Ridge Regression':
        numeric_steps.append(('scale', StandardScaler()))
    preprocess = ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ])
    if model_name == 'Ridge Regression':
        estimator = Ridge(**parameters)
    elif model_name == 'Random Forest':
        estimator = RandomForestRegressor(**parameters, random_state=RANDOM_STATE, n_jobs=-1)
    else:
        base = GradientBoostingRegressor(**parameters, random_state=RANDOM_STATE, loss='huber')
        estimator = MultiOutputRegressor(base) if multi_output else base
    return Pipeline([('preprocess', preprocess), ('model', estimator)])


PARAMETER_TASK_NAME_MAP = {
    'age_overall_level': 'age_overall',
    'disability_overall_level': 'dis_overall',
    'age_activity_level': 'age_level',
    'disability_activity_level': 'dis_level',
    'age_months12': 'age_months12',
    'disability_months12': 'months12',
    'age_days10p60gr': 'age_days',
    'disability_days10p60gr': 'days'
}


def get_best_params(task_name, model_name):

    if model_name == 'Naive baseline':
        return {}

    if task_name not in PARAMETER_TASK_NAME_MAP:
        raise KeyError(
            f'No parameter-task mapping has been defined for {task_name}. '
            f'The forecast has been stopped to prevent the model from using '
            f'unchecked default parameters.'
        )

    parameter_task_name = PARAMETER_TASK_NAME_MAP[task_name]

    matching_rows = best_params_df[
        (best_params_df['task'] == parameter_task_name) &
        (best_params_df['model'] == model_name)
    ].copy()

    if matching_rows.empty:
        raise ValueError(
            f'No tuned parameters were found for forecast task {task_name}, '
            f'parameter task {parameter_task_name}, and model {model_name}. '
            f'The forecast has been stopped instead of silently using '
            f'scikit-learn default parameters.'
        )

    parameters = (
        matching_rows
        .iloc[0]
        .drop(labels=['task', 'model'])
        .dropna()
        .to_dict()
    )

    integer_parameters = [
        'n_estimators',
        'max_depth',
        'min_samples_leaf'
    ]

    for parameter_name in integer_parameters:
        if parameter_name in parameters:
            parameters[parameter_name] = int(parameters[parameter_name])

    expected_parameters = {
        'Ridge Regression': {
            'alpha'
        },
        'Random Forest': {
            'n_estimators',
            'max_depth',
            'min_samples_leaf',
            'max_features'
        },
        'Gradient Boosting': {
            'n_estimators',
            'learning_rate',
            'max_depth',
            'min_samples_leaf'
        }
    }

    if model_name not in expected_parameters:
        raise ValueError(
            f'Unsupported model name {model_name} was supplied for '
            f'forecast task {task_name}.'
        )

    missing_parameters = (
        expected_parameters[model_name] -
        set(parameters.keys())
    )

    if missing_parameters:
        raise ValueError(
            f'Tuned parameters are incomplete for forecast task {task_name} '
            f'and model {model_name}. Missing parameters are '
            f'{sorted(missing_parameters)}.'
        )

    print(
        f'Forecast task: {task_name} | '
        f'parameter task: {parameter_task_name} | '
        f'model: {model_name} | '
        f'tuned parameters: {parameters}'
    )

    return parameters

In [39]:
EXPECTED_PARAMETER_TASKS = {
    'age_overall',
    'dis_overall',
    'age_level',
    'dis_level',
    'age_months12',
    'months12',
    'age_days',
    'days'
}

AVAILABLE_PARAMETER_TASKS = set(
    best_params_df['task']
    .dropna()
    .astype(str)
    .unique()
)

MISSING_PARAMETER_TASKS = (
    EXPECTED_PARAMETER_TASKS -
    AVAILABLE_PARAMETER_TASKS
)

if MISSING_PARAMETER_TASKS:
    raise ValueError(
        f'The hyperparameter file is missing the following tasks: '
        f'{sorted(MISSING_PARAMETER_TASKS)}'
    )

print(
    'Hyperparameter task-name check passed. '
    'All eight parameter-task names are available.'
)

Hyperparameter task-name check passed. All eight parameter-task names are available.


In [40]:
def refit_production_model(raw_frame, panel_keys, target_cols, weight_col,
                            feature_level, model_name, best_params, is_composition):
    if feature_level == 'overall':
        prepared = add_lag_features(raw_frame, panel_keys, target_cols, lags=(1, 2))
        extra_cols = [f'{c}_lag2' for c in target_cols]
    else:
        prepared = add_lag_features(raw_frame, panel_keys, target_cols, lags=(1,), rolling_window=2)
        extra_cols = [f'{c}_roll2' for c in target_cols]

    lag1_cols = [f'{c}_lag1' for c in target_cols]
    numeric = lag1_cols + extra_cols + ['time_trend', 'is_covid_year']
    categorical = list(panel_keys)

    train_all = prepared.dropna(subset=target_cols + [weight_col] + lag1_cols + extra_cols).copy()

    if model_name == 'Ridge Regression':
        train_all, interaction_cols = add_interaction_terms(train_all, panel_keys[1])
        numeric = numeric + interaction_cols

    model = build_model(model_name, best_params, numeric, categorical, multi_output=is_composition)
    weights = train_all[weight_col].to_numpy()
    y = shares_to_alr(train_all[target_cols]) if is_composition else train_all[target_cols[0]]

    model.fit(train_all[numeric + categorical], y, model__sample_weight=weights)
    return model

In [41]:
def build_forecast_seed(raw_frame, panel_keys, target_cols, require_full_history,
                         start_year=FORECAST_START_YEAR):
    two_years = raw_frame[
        raw_frame['year'].isin([start_year - 1, start_year])
    ].dropna(subset=target_cols)

    year_counts = two_years.groupby(panel_keys)['year'].nunique()
    full_history_keys = set(year_counts[year_counts == 2].index)
    partial_keys = set(year_counts[year_counts == 1].index)
    has_year8 = set(
        two_years[two_years['year'] == start_year].set_index(panel_keys).index
    )
    year8_only_keys = partial_keys & has_year8
    dropped_no_year8 = partial_keys - has_year8

    if require_full_history:
        keep_keys = full_history_keys
        dropped_no_year7 = year8_only_keys
    else:
        keep_keys = full_history_keys | year8_only_keys
        dropped_no_year7 = set()

    seed = two_years[two_years.set_index(panel_keys).index.isin(keep_keys)].copy()
    diagnostic = pd.DataFrame([{
        'panels_full_2yr_history': len(full_history_keys & keep_keys),
        'panels_year8_only_used': len(year8_only_keys & keep_keys),
        'panels_dropped_no_year7': len(dropped_no_year7),
        'panels_dropped_no_year8': len(dropped_no_year8),
    }])
    return seed, diagnostic

In [42]:
def add_interaction_terms(frame, group_col, time_col='time_trend'):
    dummies = pd.get_dummies(frame[group_col], prefix=f'{group_col}_x_time')
    interaction_cols = list(dummies.columns)
    frame = frame.copy()
    frame[interaction_cols] = dummies.mul(frame[time_col], axis=0)
    return frame, interaction_cols


def forecast_composition_task(fitted_model, seed_two_years, panel_keys, target_cols,
                               feature_level, needs_interaction,
                               start_year=FORECAST_START_YEAR,
                               horizon=FORECAST_HORIZON, year_ref=YEAR_REF):
    lag1_cols = [f'{c}_lag1' for c in target_cols]
    extra_cols = ([f'{c}_lag2' for c in target_cols] if feature_level == 'overall'
                  else [f'{c}_roll2' for c in target_cols])

    history = {}
    for key, grp in seed_two_years.sort_values('year').groupby(panel_keys):
        history[key] = grp[target_cols].to_numpy().tolist()
    keys = list(history.keys())
    records = []

    for step in range(1, horizon + 1):
        forecast_year = start_year + step
        rows = []
        for key in keys:
            vals = history[key]
            last = vals[-1]
            prev = vals[-2] if len(vals) >= 2 else None
            row = dict(zip(panel_keys, key))
            for i, c in enumerate(target_cols):
                row[f'{c}_lag1'] = last[i]
                if feature_level == 'overall':
                    row[f'{c}_lag2'] = prev[i] if prev is not None else last[i]
                else:
                    row[f'{c}_roll2'] = (last[i] + prev[i]) / 2.0 if prev is not None else last[i]
            row['time_trend'] = forecast_year / year_ref
            row['is_covid_year'] = 0
            rows.append(row)
        features = pd.DataFrame(rows)

        numeric = lag1_cols + extra_cols + ['time_trend', 'is_covid_year']
        model_input = features
        cols_needed = numeric + panel_keys
        if needs_interaction:
            model_input, interaction_cols = add_interaction_terms(features, panel_keys[1])
            cols_needed = numeric + interaction_cols + panel_keys

        if fitted_model is None:
            predicted_shares = features[lag1_cols].to_numpy()
        else:
            predicted_alr = fitted_model.predict(model_input[cols_needed])
            predicted_shares = alr_to_shares(predicted_alr)

        step_result = features[panel_keys].copy()
        step_result['year'] = forecast_year
        for i, c in enumerate(target_cols):
            step_result[c] = predicted_shares[:, i]
        records.append(step_result)

        for idx, key in enumerate(keys):
            new_val = predicted_shares[idx].tolist()
            history[key] = [history[key][-1], new_val] if len(history[key]) >= 1 else [new_val]

    return pd.concat(records, ignore_index=True)

In [43]:
def forecast_single_rate_task(fitted_model, seed_two_years, panel_keys, target_col,
                               feature_level, needs_interaction,
                               start_year=FORECAST_START_YEAR,
                               horizon=FORECAST_HORIZON, year_ref=YEAR_REF):
    lag1_col = f'{target_col}_lag1'
    extra_col = f'{target_col}_lag2' if feature_level == 'overall' else f'{target_col}_roll2'

    history = {}
    for key, grp in seed_two_years.sort_values('year').groupby(panel_keys):
        history[key] = grp[target_col].to_numpy().tolist()
    keys = list(history.keys())
    records = []

    for step in range(1, horizon + 1):
        forecast_year = start_year + step
        rows = []
        for key in keys:
            vals = history[key]
            last = vals[-1]
            prev = vals[-2] if len(vals) >= 2 else None
            row = dict(zip(panel_keys, key))
            row[lag1_col] = last
            if feature_level == 'overall':
                row[extra_col] = prev if prev is not None else last
            else:
                row[extra_col] = (last + prev) / 2.0 if prev is not None else last
            row['time_trend'] = forecast_year / year_ref
            row['is_covid_year'] = 0
            rows.append(row)
        features = pd.DataFrame(rows)

        numeric = [lag1_col, extra_col, 'time_trend', 'is_covid_year']
        model_input = features
        cols_needed = numeric + panel_keys
        if needs_interaction:
            model_input, interaction_cols = add_interaction_terms(features, panel_keys[1])
            cols_needed = numeric + interaction_cols + panel_keys

        if fitted_model is None:
            predicted = features[lag1_col].to_numpy()
        else:
            predicted = np.clip(fitted_model.predict(model_input[cols_needed]), 0, 1)

        step_result = features[panel_keys].copy()
        step_result['year'] = forecast_year
        step_result[target_col] = predicted
        records.append(step_result)

        for idx, key in enumerate(keys):
            new_val = predicted[idx]
            history[key] = [history[key][-1], new_val] if len(history[key]) >= 1 else [new_val]

    return pd.concat(records, ignore_index=True)

## 2. Forecast: age overall activity level


In [44]:
age_overall_raw = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_overall_raw['LA_2023'] = age_overall_raw['LA_2023'].astype('Int64').astype(str)

age_overall_targets = ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate']
age_overall_panel_keys = ['LA_2023', 'age_group']

age_overall_best_model_name = select_best_model(summary, 'age_overall_level', 'total_variation')
print('best model, age overall level:', age_overall_best_model_name)

if age_overall_best_model_name == 'Naive baseline':
    age_overall_production_model = None
else:
    age_overall_params = get_best_params('age_overall_level', age_overall_best_model_name)
    age_overall_production_model = refit_production_model(
        age_overall_raw, age_overall_panel_keys, age_overall_targets,
        'weighted_n_overall_activity_level',
        feature_level='overall', model_name=age_overall_best_model_name,
        best_params=age_overall_params, is_composition=True,
    )
age_overall_needs_interaction = (age_overall_best_model_name == 'Ridge Regression')

age_overall_seed, age_overall_seed_diag = build_forecast_seed(
    age_overall_raw, age_overall_panel_keys, age_overall_targets, require_full_history=True,
)
print('seed completeness, age overall level:')
print(age_overall_seed_diag)

age_overall_forecast = forecast_composition_task(
    age_overall_production_model, age_overall_seed,
    panel_keys=age_overall_panel_keys, target_cols=age_overall_targets,
    feature_level='overall', needs_interaction=age_overall_needs_interaction,
)
age_overall_forecast_naive = forecast_composition_task(
    None, age_overall_seed,
    panel_keys=age_overall_panel_keys, target_cols=age_overall_targets,
    feature_level='overall', needs_interaction=False,
)
age_overall_forecast.head()

best model, age overall level: Gradient Boosting
Forecast task: age_overall_level | parameter task: age_overall | model: Gradient Boosting | tuned parameters: {'n_estimators': 120, 'max_depth': 2, 'min_samples_leaf': 15, 'learning_rate': 0.05}
seed completeness, age overall level:
   panels_full_2yr_history  panels_year8_only_used  panels_dropped_no_year7  \
0                      256                       0                        0   

   panels_dropped_no_year8  
0                        0  


,LA_2023,age_group,year,overall_inactive_rate,overall_fairly_active_rate,overall_active_rate
0,107,16-24,9,0.265495,0.092362,0.642143
1,107,25-34,9,0.188522,0.097815,0.713663
2,107,35-44,9,0.234714,0.104495,0.660791
3,107,45-54,9,0.232826,0.103067,0.664106
4,107,55-64,9,0.242567,0.098737,0.658696


## 3. Load the shared activity-level panel


In [45]:
age_activity_raw = pd.read_csv(AGE_DATA_DIR / 'q3_age_activity_participation_level_panel_complete.csv')
age_activity_raw['LA_2023'] = age_activity_raw['LA_2023'].astype('Int64').astype(str)
age_activity_panel_keys = ['LA_2023', 'age_group', 'activity_suffix']
age_activity_raw.head()

,year,LA_2023,age_group,activity_suffix,months12_available,days10p60gr_available,activity_level_available,months12_rate,n_months12,weighted_n_months12,...,n_days10p60gr,weighted_n_days10p60gr,activity_inactive_rate,activity_fairly_active_rate,activity_active_rate,n_activity_level,weighted_n_activity_level,small_cell_months12,small_cell_days10p60gr,small_cell_activity_level
0,1,8,16-24,ABSEILING_H03,True,True,True,0.000000,104,102.488535,...,104,102.488535,1.000000,0.000000,0.00000,104,102.488535,False,False,False
1,1,8,16-24,ACTTRAV_C03,True,True,True,0.815521,104,102.488535,...,104,102.488535,0.627589,0.156891,0.21552,104,102.488535,False,False,False
2,1,8,16-24,AIKIDO_S04,True,True,True,0.000000,74,74.596065,...,74,74.596065,1.000000,0.000000,0.00000,104,102.488535,False,False,False
3,1,8,16-24,AIRGUN_S08,True,True,True,0.000000,74,74.596065,...,74,74.596065,1.000000,0.000000,0.00000,104,102.488535,False,False,False
4,1,8,16-24,ARCHERY_J01,True,True,True,0.021049,104,102.488535,...,104,102.488535,1.000000,0.000000,0.00000,104,102.488535,False,False,False


## 4. Forecast: age activity-specific level


In [46]:
age_level_targets = ['activity_inactive_rate', 'activity_fairly_active_rate', 'activity_active_rate']

age_level_best_model_name = select_best_model(summary, 'age_activity_level', 'total_variation')
print('best model, age activity level:', age_level_best_model_name)

if age_level_best_model_name == 'Naive baseline':
    age_level_production_model = None
else:
    age_level_params = get_best_params('age_activity_level', age_level_best_model_name)
    age_level_production_model = refit_production_model(
        age_activity_raw, age_activity_panel_keys, age_level_targets,
        'weighted_n_activity_level',
        feature_level='activity', model_name=age_level_best_model_name,
        best_params=age_level_params, is_composition=True,
    )
age_level_needs_interaction = (age_level_best_model_name == 'Ridge Regression')

age_level_seed, age_level_seed_diag = build_forecast_seed(
    age_activity_raw, age_activity_panel_keys, age_level_targets, require_full_history=False,
)
print('seed completeness, age activity level:')
print(age_level_seed_diag)

age_level_forecast = forecast_composition_task(
    age_level_production_model, age_level_seed,
    panel_keys=age_activity_panel_keys, target_cols=age_level_targets,
    feature_level='activity', needs_interaction=age_level_needs_interaction,
)
age_level_forecast_naive = forecast_composition_task(
    None, age_level_seed,
    panel_keys=age_activity_panel_keys, target_cols=age_level_targets,
    feature_level='activity', needs_interaction=False,
)
age_level_forecast.head()

best model, age activity level: Gradient Boosting
Forecast task: age_activity_level | parameter task: age_level | model: Gradient Boosting | tuned parameters: {'n_estimators': 120, 'max_depth': 2, 'min_samples_leaf': 15, 'learning_rate': 0.05}
seed completeness, age activity level:
   panels_full_2yr_history  panels_year8_only_used  panels_dropped_no_year7  \
0                    31585                     106                        0   

   panels_dropped_no_year8  
0                       53  


,LA_2023,age_group,activity_suffix,year,activity_inactive_rate,activity_fairly_active_rate,activity_active_rate
0,107,16-24,ABSEILING_H03,9,0.999998,0.000001,0.000001
1,107,16-24,ACTTRAV_C03,9,0.639287,0.146256,0.214457
2,107,16-24,AIKIDO_S04,9,0.999998,0.000001,0.000001
3,107,16-24,AIRGUN_S08,9,0.999998,0.000001,0.000001
4,107,16-24,ARCHERY_J01,9,0.999998,0.000001,0.000001


## 5. Forecast: MONTHS_12 and DAYS10P60GR participation rate


In [47]:
age_months12_best_model_name = select_best_model(summary, 'age_months12', 'months12_rate_mae')
print('best model, age MONTHS_12:', age_months12_best_model_name)

if age_months12_best_model_name == 'Naive baseline':
    age_months12_production_model = None
else:
    age_months12_params = get_best_params('age_months12', age_months12_best_model_name)
    age_months12_production_model = refit_production_model(
        age_activity_raw, age_activity_panel_keys, ['months12_rate'], 'weighted_n_months12',
        feature_level='activity', model_name=age_months12_best_model_name,
        best_params=age_months12_params, is_composition=False,
    )
age_months12_needs_interaction = (age_months12_best_model_name == 'Ridge Regression')

age_months12_seed, age_months12_seed_diag = build_forecast_seed(
    age_activity_raw, age_activity_panel_keys, ['months12_rate'], require_full_history=False,
)
print('seed completeness, age MONTHS_12:')
print(age_months12_seed_diag)

age_months12_forecast = forecast_single_rate_task(
    age_months12_production_model, age_months12_seed,
    panel_keys=age_activity_panel_keys, target_col='months12_rate',
    feature_level='activity', needs_interaction=age_months12_needs_interaction,
)
age_months12_forecast_naive = forecast_single_rate_task(
    None, age_months12_seed,
    panel_keys=age_activity_panel_keys, target_col='months12_rate',
    feature_level='activity', needs_interaction=False,
)

age_days_best_model_name = select_best_model(summary, 'age_days10p60gr', 'days10p60gr_rate_mae')
print('best model, age DAYS10P60GR:', age_days_best_model_name)

if age_days_best_model_name == 'Naive baseline':
    age_days_production_model = None
else:
    age_days_params = get_best_params('age_days10p60gr', age_days_best_model_name)
    age_days_production_model = refit_production_model(
        age_activity_raw, age_activity_panel_keys, ['days10p60gr_rate'], 'weighted_n_days10p60gr',
        feature_level='activity', model_name=age_days_best_model_name,
        best_params=age_days_params, is_composition=False,
    )
age_days_needs_interaction = (age_days_best_model_name == 'Ridge Regression')

age_days_seed, age_days_seed_diag = build_forecast_seed(
    age_activity_raw, age_activity_panel_keys, ['days10p60gr_rate'], require_full_history=False,
)
print('seed completeness, age DAYS10P60GR:')
print(age_days_seed_diag)

age_days_forecast = forecast_single_rate_task(
    age_days_production_model, age_days_seed,
    panel_keys=age_activity_panel_keys, target_col='days10p60gr_rate',
    feature_level='activity', needs_interaction=age_days_needs_interaction,
)
age_days_forecast_naive = forecast_single_rate_task(
    None, age_days_seed,
    panel_keys=age_activity_panel_keys, target_col='days10p60gr_rate',
    feature_level='activity', needs_interaction=False,
)

age_months12_forecast.head()

best model, age MONTHS_12: Naive baseline
seed completeness, age MONTHS_12:
   panels_full_2yr_history  panels_year8_only_used  panels_dropped_no_year7  \
0                    31585                     106                        0   

   panels_dropped_no_year8  
0                       53  
best model, age DAYS10P60GR: Ridge Regression
Forecast task: age_days10p60gr | parameter task: age_days | model: Ridge Regression | tuned parameters: {'alpha': 100.0}
seed completeness, age DAYS10P60GR:
   panels_full_2yr_history  panels_year8_only_used  panels_dropped_no_year7  \
0                    31585                     106                        0   

   panels_dropped_no_year8  
0                       53  


,LA_2023,age_group,activity_suffix,year,months12_rate
0,107,16-24,ABSEILING_H03,9,0.000000
1,107,16-24,ACTTRAV_C03,9,0.796338
2,107,16-24,AIKIDO_S04,9,0.000000
3,107,16-24,AIRGUN_S08,9,0.000000
4,107,16-24,ARCHERY_J01,9,0.000000


## 6. Top two predicted activities and ranking margin, by forecast year

In [48]:
age_overall_raw = pd.read_csv(AGE_DATA_DIR / 'q3_age_overall_activity_level_panel.csv')
age_activity_raw = pd.read_csv(AGE_DATA_DIR / 'q3_age_activity_participation_level_panel_complete.csv')
age_overall_raw['LA_2023'] = age_overall_raw['LA_2023'].astype('Int64').astype(str)
age_activity_raw['LA_2023'] = age_activity_raw['LA_2023'].astype('Int64').astype(str)

age_overall_panel_keys = ['LA_2023', 'age_group']
age_activity_panel_keys = ['LA_2023', 'age_group', 'activity_suffix']
age_overall_targets = ['overall_inactive_rate', 'overall_fairly_active_rate', 'overall_active_rate']
age_level_targets = ['activity_inactive_rate', 'activity_fairly_active_rate', 'activity_active_rate']

historical_diagnostic_columns = ['historical_years_available', 'historical_small_cell_years', 'historical_median_n', 'historical_min_n', 'historical_small_cell_rate', 'historical_small_cell_majority', 'historical_incomplete_outcome_history', 'historical_reliability_warning']

def load_clean_forecast(filename):
    forecast = pd.read_csv(FORECAST_DIR / filename).drop(columns=historical_diagnostic_columns, errors='ignore')
    forecast['LA_2023'] = forecast['LA_2023'].astype('Int64').astype(str)
    return forecast

age_overall_forecast = load_clean_forecast('forecast_age_overall_level.csv')
age_overall_forecast_naive = load_clean_forecast('forecast_age_overall_level_naive.csv')
age_level_forecast = load_clean_forecast('forecast_age_activity_level.csv')
age_level_forecast_naive = load_clean_forecast('forecast_age_activity_level_naive.csv')
age_months12_forecast = load_clean_forecast('forecast_age_months12.csv')
age_months12_forecast_naive = load_clean_forecast('forecast_age_months12_naive.csv')
age_days_forecast = load_clean_forecast('forecast_age_days10p60gr.csv')
age_days_forecast_naive = load_clean_forecast('forecast_age_days10p60gr_naive.csv')

assert age_activity_raw['activity_suffix'].nunique() == 124
assert set(age_months12_forecast['year']) == set(range(9, 17))

print('Existing Year 9-16 forecasts loaded successfully.')
print('Rows per activity forecast year:', age_months12_forecast.groupby('year').size().to_dict())

Existing Year 9-16 forecasts loaded successfully.
Rows per activity forecast year: {9: 31691, 10: 31691, 11: 31691, 12: 31691, 13: 31691, 14: 31691, 15: 31691, 16: 31691}


In [49]:
def top_two_activities_by_year(forecast_frame, id_cols, activity_col, value_col):
    ranked = forecast_frame.sort_values(id_cols + [value_col, activity_col], ascending=[True] * len(id_cols) + [False, True]).copy()
    ranked['activity_rank'] = ranked.groupby(id_cols).cumcount() + 1
    ranked = ranked[ranked['activity_rank'] <= 2].copy()

    top_activity = ranked[ranked['activity_rank'] == 1][id_cols + [activity_col, value_col]].copy()
    second_activity = ranked[ranked['activity_rank'] == 2][id_cols + [activity_col, value_col]].copy()
    second_activity = second_activity.rename(columns={activity_col: 'second_activity_suffix', value_col: 'second_months12_rate'})

    result = top_activity.merge(second_activity, on=id_cols, how='left', validate='one_to_one')
    result['ranking_margin'] = result[value_col] - result['second_months12_rate']
    result['top_two_tied'] = np.isclose(result[value_col], result['second_months12_rate'], atol=1e-10)
    return result

age_top_activity_forecast = top_two_activities_by_year(age_months12_forecast, id_cols=['year', 'LA_2023', 'age_group'], activity_col='activity_suffix', value_col='months12_rate')

assert age_top_activity_forecast.duplicated(subset=['year', 'LA_2023', 'age_group']).sum() == 0
assert age_top_activity_forecast['second_activity_suffix'].notna().all()
assert (age_top_activity_forecast['ranking_margin'] >= -1e-12).all()

print('Top-two rows:', len(age_top_activity_forecast))
print('Smallest ranking margins:')
display(age_top_activity_forecast.sort_values('ranking_margin').head(10))

Top-two rows: 2048
Smallest ranking margins:


,year,LA_2023,age_group,activity_suffix,months12_rate,second_activity_suffix,second_months12_rate,ranking_margin,top_two_tied
1150,13,171,75-84,ACTTRAV_C03,0.579866,WALKTRAV_B02,0.579866,0.0,True
14,9,109,75-84,ACTTRAV_C03,0.345443,WALKTRAV_B02,0.345443,0.0,True
2017,16,78,25-34,ACTTRAV_C03,0.698606,WALKTRAV_B02,0.698606,0.0,True
1455,14,279,85+,ACTTRAV_C03,0.527153,WALKTRAV_B02,0.527153,0.0,True
0,9,107,16-24,ACTTRAV_C03,0.796338,WALKTRAV_B02,0.796338,0.0,True
1791,15,91,85+,ACTTRAV_C03,0.395405,WALKTRAV_B02,0.395405,0.0,True
759,11,9,85+,ACTTRAV_C03,0.589443,WALKTRAV_B02,0.589443,0.0,True
35,9,117,45-54,ACTTRAV_C03,0.699249,WALKTRAV_B02,0.699249,0.0,True
33,9,117,25-34,ACTTRAV_C03,0.658870,WALKTRAV_B02,0.658870,0.0,True
2027,16,8,45-54,ACTTRAV_C03,0.616315,WALKTRAV_B02,0.616315,0.0,True


## 7. Historical sample-size warnings for future forecasts

In [50]:
def historical_sample_diagnostics(frame, keys, target_col, n_col):
    expected_years = frame["year"].nunique()

    valid = frame.dropna(
        subset=[target_col, n_col]
    ).copy()

    valid["historical_small_cell"] = valid[n_col] < 30

    result = valid.groupby(keys).agg(
        historical_years_available=(target_col, "size"),
        historical_small_cell_years=("historical_small_cell", "sum"),
        historical_median_n=(n_col, "median"),
        historical_min_n=(n_col, "min")
    ).reset_index()

    result["historical_small_cell_rate"] = (
        result["historical_small_cell_years"] /
        result["historical_years_available"]
    )

    result["historical_small_cell_majority"] = (
        result["historical_small_cell_rate"] >= 0.5
    )

    result["historical_incomplete_outcome_history"] = (
        result["historical_years_available"] < expected_years
    )

    result["historical_reliability_warning"] = (
        result["historical_small_cell_majority"] |
        result["historical_incomplete_outcome_history"]
    )

    return result


age_overall_reliability = historical_sample_diagnostics(
    age_overall_raw,
    age_overall_panel_keys,
    "overall_active_rate",
    "n_overall_activity_level"
)

age_level_reliability = historical_sample_diagnostics(
    age_activity_raw,
    age_activity_panel_keys,
    "activity_active_rate",
    "n_activity_level"
)

age_months12_reliability = historical_sample_diagnostics(
    age_activity_raw,
    age_activity_panel_keys,
    "months12_rate",
    "n_months12"
)

age_days_reliability = historical_sample_diagnostics(
    age_activity_raw,
    age_activity_panel_keys,
    "days10p60gr_rate",
    "n_days10p60gr"
)


age_overall_forecast = age_overall_forecast.merge(
    age_overall_reliability,
    on=age_overall_panel_keys,
    how="left"
)

age_overall_forecast_naive = age_overall_forecast_naive.merge(
    age_overall_reliability,
    on=age_overall_panel_keys,
    how="left"
)

age_level_forecast = age_level_forecast.merge(
    age_level_reliability,
    on=age_activity_panel_keys,
    how="left"
)

age_level_forecast_naive = age_level_forecast_naive.merge(
    age_level_reliability,
    on=age_activity_panel_keys,
    how="left"
)

age_months12_forecast = age_months12_forecast.merge(
    age_months12_reliability,
    on=age_activity_panel_keys,
    how="left"
)

age_months12_forecast_naive = age_months12_forecast_naive.merge(
    age_months12_reliability,
    on=age_activity_panel_keys,
    how="left"
)

age_days_forecast = age_days_forecast.merge(
    age_days_reliability,
    on=age_activity_panel_keys,
    how="left"
)

age_days_forecast_naive = age_days_forecast_naive.merge(
    age_days_reliability,
    on=age_activity_panel_keys,
    how="left"
)

age_top_activity_forecast = age_top_activity_forecast.merge(
    age_months12_reliability,
    on=age_activity_panel_keys,
    how="left"
)


age_overall_reliability.to_csv(
    FORECAST_DIR / "historical_sample_diagnostics_age_overall.csv",
    index=False
)

age_level_reliability.to_csv(
    FORECAST_DIR / "historical_sample_diagnostics_age_activity_level.csv",
    index=False
)

age_months12_reliability.to_csv(
    FORECAST_DIR / "historical_sample_diagnostics_age_months12.csv",
    index=False
)

age_days_reliability.to_csv(
    FORECAST_DIR / "historical_sample_diagnostics_age_days10p60gr.csv",
    index=False
)


age_warning_summary = pd.DataFrame({
    "task": [
        "overall",
        "activity_level",
        "months12",
        "days10p60gr",
        "top_activity"
    ],
    "warning_rate": [
        age_overall_forecast["historical_reliability_warning"].mean(),
        age_level_forecast["historical_reliability_warning"].mean(),
        age_months12_forecast["historical_reliability_warning"].mean(),
        age_days_forecast["historical_reliability_warning"].mean(),
        age_top_activity_forecast["historical_reliability_warning"].mean()
    ]
})

age_warning_summary.to_csv(
    FORECAST_DIR / "forecast_age_historical_warning_summary.csv",
    index=False
)

print(age_warning_summary.round(3).to_string(index=False))

          task  warning_rate
       overall         0.215
activity_level         0.230
      months12         0.263
   days10p60gr         0.263
  top_activity         0.215


## 8. Sanity checks on forecasts

In [51]:
row_counts = age_level_forecast.groupby('year').size()
print('rows per forecast year, activity level:')
print(row_counts)

dup_check = {
    'age_overall_level': age_overall_forecast.duplicated(subset=age_overall_panel_keys + ['year']).sum(),
    'age_activity_level': age_level_forecast.duplicated(subset=age_activity_panel_keys + ['year']).sum(),
    'age_months12': age_months12_forecast.duplicated(subset=age_activity_panel_keys + ['year']).sum(),
    'age_days10p60gr': age_days_forecast.duplicated(subset=age_activity_panel_keys + ['year']).sum(),
}
print('duplicate panel x year rows (should all be 0):', dup_check)

range_check = {
    'age_overall_level': (age_overall_forecast[age_overall_targets].min().min(),
                           age_overall_forecast[age_overall_targets].max().max()),
    'age_activity_level': (age_level_forecast[age_level_targets].min().min(),
                            age_level_forecast[age_level_targets].max().max()),
    'age_months12': (age_months12_forecast['months12_rate'].min(),
                      age_months12_forecast['months12_rate'].max()),
    'age_days10p60gr': (age_days_forecast['days10p60gr_rate'].min(),
                         age_days_forecast['days10p60gr_rate'].max()),
}
print('min/max, should stay within 0 to 1:', range_check)

mems7gr_sum_check = (age_overall_forecast[age_overall_targets].sum(axis=1) - 1).abs().max()
print('max deviation from summing to 1, overall level:', mems7gr_sum_check)
mems7gr_sum_check_level = (age_level_forecast[age_level_targets].sum(axis=1) - 1).abs().max()
print('max deviation from summing to 1, activity level:', mems7gr_sum_check_level)

year9_vs_year8 = age_overall_raw[age_overall_raw['year'] == FORECAST_START_YEAR].merge(
    age_overall_forecast[age_overall_forecast['year'] == FORECAST_START_YEAR + 1],
    on=age_overall_panel_keys, suffixes=('_year8', '_year9'),
)
year9_vs_year8['active_rate_jump'] = (
    year9_vs_year8['overall_active_rate_year9'] - year9_vs_year8['overall_active_rate_year8']
).abs()
print('largest active_rate jump from Year 8 to Year 9:')
print(year9_vs_year8.sort_values('active_rate_jump', ascending=False).head(5)[
    age_overall_panel_keys + ['overall_active_rate_year8', 'overall_active_rate_year9', 'active_rate_jump']
])

top_activity_counts = age_top_activity_forecast.groupby('year')['activity_suffix'].nunique()
print('number of distinct activities selected as top, by forecast year:')
print(top_activity_counts)

comparison = age_overall_forecast.merge(
    age_overall_forecast_naive,
    on=['LA_2023', 'age_group', 'year'],
    suffixes=('_model', '_naive'),
)
comparison['active_rate_gap'] = (
    comparison['overall_active_rate_model'] - comparison['overall_active_rate_naive']
).abs()
gap_by_year = comparison.groupby('year')['active_rate_gap'].mean()
print('mean |model - naive| gap in active_rate, by forecast year:')
print(gap_by_year.round(4))

rows per forecast year, activity level:
year
9     31691
10    31691
11    31691
12    31691
13    31691
14    31691
15    31691
16    31691
dtype: int64
duplicate panel x year rows (should all be 0): {'age_overall_level': np.int64(0), 'age_activity_level': np.int64(0), 'age_months12': np.int64(0), 'age_days10p60gr': np.int64(0)}
min/max, should stay within 0 to 1: {'age_overall_level': (0.0001473084910606, 0.8593980841611306), 'age_activity_level': (1.4125993023304864e-07, 0.9999991374855838), 'age_months12': (0.0, 1.0), 'age_days10p60gr': (0.0, 0.6347474015157072)}
max deviation from summing to 1, overall level: 3.3306690738754696e-16
max deviation from summing to 1, activity level: 4.440892098500626e-16
largest active_rate jump from Year 8 to Year 9:
    LA_2023 age_group  overall_active_rate_year8  overall_active_rate_year9  \
247     280       85+                   1.000000                   0.487122   
7         8       85+                   1.000000                   0.491238   

## 9. Save forecast outputs

In [52]:
age_overall_forecast.to_csv(FORECAST_DIR / 'forecast_age_overall_level.csv', index=False)
age_overall_forecast_naive.to_csv(FORECAST_DIR / 'forecast_age_overall_level_naive.csv', index=False)
age_level_forecast.to_csv(FORECAST_DIR / 'forecast_age_activity_level.csv', index=False)
age_level_forecast_naive.to_csv(FORECAST_DIR / 'forecast_age_activity_level_naive.csv', index=False)
age_months12_forecast.to_csv(FORECAST_DIR / 'forecast_age_months12.csv', index=False)
age_months12_forecast_naive.to_csv(FORECAST_DIR / 'forecast_age_months12_naive.csv', index=False)
age_days_forecast.to_csv(FORECAST_DIR / 'forecast_age_days10p60gr.csv', index=False)
age_days_forecast_naive.to_csv(FORECAST_DIR / 'forecast_age_days10p60gr_naive.csv', index=False)
age_top_activity_forecast.to_csv(FORECAST_DIR / 'forecast_age_top_activity_by_year.csv', index=False)

age_seed_diagnostics = pd.concat([
    age_overall_seed_diag.assign(task='age_overall_level'),
    age_level_seed_diag.assign(task='age_activity_level'),
    age_months12_seed_diag.assign(task='age_months12'),
    age_days_seed_diag.assign(task='age_days10p60gr'),
], ignore_index=True)
age_seed_diagnostics.to_csv(FORECAST_DIR / 'age_seed_completeness_diagnostics.csv', index=False)

print('Saved to', FORECAST_DIR)

Saved to C:\Users\Lenovo\Documents\GitHub\London-Sport2-Group20\Jingyi Hua\outputs\forecast_year9_16
